# ADHD-200 fMRI strict cross-site benchmark

This notebook evaluates whether resting-state fMRI adds reproducible ADHD signal beyond age, sex, scan quality, and motion. The primary cohort excludes Pittsburgh because its available A424 cohort is single-class. Pittsburgh remains documented as a sensitivity cohort.

Evaluation uses nested leave-one-site-out (LOSO). Every imputer, feature selector, scaler, and hyperparameter is fitted using training sites only. Primary metric: macro-average of evaluable held-out-site AUCs. Weighted macro and pooled out-of-fold AUC are secondary.

Research use only. This is not a clinical diagnostic system.

In [ ]:
%pip install -q scikit-learn scipy requests

In [ ]:
from google.colab import drive
drive.mount('<DRIVE_MOUNT>')

from pathlib import Path
from io import StringIO
from concurrent.futures import ThreadPoolExecutor, as_completed
import re, warnings
import numpy as np
import pandas as pd
import requests
from scipy.signal import periodogram
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=RuntimeWarning)
ROOT=Path('<DATA_DIR>')
BRAINLM=ROOT/'fmri'/'brainlm_a424'
TS_DIR=BRAINLM/'timeseries_raw'
OUT=ROOT/'fmri'/'strict_loso_benchmark'
OUT.mkdir(parents=True,exist_ok=True)
INDEX=BRAINLM/'fmriprep_aroma_index.csv'
META=BRAINLM/'brainlm_subject_embedding_metadata.csv'
print('Output:',OUT)

## 1. Locked cohort and covariates

The primary comparison uses the same 409 A424 subjects across all imaging models. Age and sex come from the existing strict structural manifest. Coarse resting-state QC comes from the original site phenotypic tables. Continuous motion summaries are downloaded from each matching fMRIPrep confounds file and cached.

In [ ]:
meta=pd.read_csv(META,dtype={'subject_id':str})
meta['subject_id']=meta.subject_id.astype(str)
if not meta.subject_id.str.startswith('sub-').all():
    meta['subject_id']='sub-'+meta.subject_id.str.replace('sub-','',regex=False)

dem=pd.read_csv(ROOT/'manifest_all_sitefirst_strictpass_noBrown.csv')
dem=dem.rename(columns={'sub_id':'subject_id'})[['subject_id','age','sex']]
dem['subject_id']=dem.subject_id.astype(str)
if not dem.subject_id.str.startswith('sub-').all():
    dem['subject_id']='sub-'+dem.subject_id.str.replace('sub-','',regex=False)
dem=dem.drop_duplicates('subject_id',keep='first')

phen=[]
for name in ['KKI','NYU','NeuroIMAGE','OHSU','Peking_1','Pittsburgh']:
    p=ROOT/f'{name}_phenotypic.csv'
    if not p.exists():
        continue
    d=pd.read_csv(p)
    idcol='ScanDir ID' if 'ScanDir ID' in d.columns else 'ID'
    d['subject_id']='sub-'+d[idcol].astype(str).str.replace(r'\.0$','',regex=True).str.zfill(7)
    keep=['subject_id']+[c for c in ['Full4 IQ','Full2 IQ','QC_Rest_1','ADHD Index','Inattentive','Hyper/Impulsive'] if c in d]
    phen.append(d[keep])
phen=pd.concat(phen,ignore_index=True).drop_duplicates('subject_id')
phen['iq']=pd.to_numeric(phen.get('Full4 IQ'),errors='coerce')
phen['iq']=phen['iq'].fillna(pd.to_numeric(phen.get('Full2 IQ'),errors='coerce'))
phen['qc_rest']=pd.to_numeric(phen.get('QC_Rest_1'),errors='coerce')

cohort=meta.merge(dem,on='subject_id',how='left').merge(
    phen[['subject_id','iq','qc_rest','ADHD Index','Inattentive','Hyper/Impulsive']],
    on='subject_id',how='left')
cohort['sex_male']=cohort.sex.astype(str).str.lower().map({'male':1,'female':0,'1':1,'0':0})
cohort['label']=cohort.label.astype(int)
cohort['primary']=cohort.site.ne('Pittsburgh')
cohort.to_csv(OUT/'locked_cohort_all_445.csv',index=False)
primary=cohort[cohort.primary].reset_index(drop=True)
primary.to_csv(OUT/'locked_primary_cohort_409.csv',index=False)

display(cohort.groupby(['site','label']).size().rename('n').reset_index())
display(primary.groupby('site')[['age','sex_male','iq','qc_rest']].agg(['count','mean']))
print('All A424:',len(cohort),'Primary:',len(primary),'Sites:',primary.site.nunique())

In [ ]:
# Continuous motion summaries from matching fMRIPrep confounds files.
MOTION=OUT/'motion_qc.csv'
BASE='https://fcp-indi.s3.amazonaws.com/'
idx=pd.read_csv(INDEX,dtype={'subject_id':str})
idx=idx[idx.subject_id.isin(cohort.subject_id)].copy()

def confounds_key(bold_key):
    folder,name=bold_key.rsplit('/',1)
    stem=re.sub(r'_space-[^_]+_desc-smoothAROMAnonaggr_bold\.nii\.gz$','',name)
    return f'{folder}/{stem}_desc-confounds_regressors.tsv'

def motion_one(row):
    out={'subject_id':row.subject_id}
    try:
        key=confounds_key(row.s3_key)
        r=requests.get(BASE+key,timeout=30)
        r.raise_for_status()
        d=pd.read_csv(StringIO(r.text),sep='\t')
        fd=pd.to_numeric(d.get('framewise_displacement'),errors='coerce').dropna().to_numpy()
        dv=pd.to_numeric(d.get('std_dvars',d.get('dvars')),errors='coerce').dropna().to_numpy()
        out.update(mean_fd=float(np.mean(fd)),max_fd=float(np.max(fd)),
                   pct_fd_gt_0p2=float(np.mean(fd>0.2)),mean_dvars=float(np.mean(dv)),
                   n_volumes=len(d),motion_status='ok')
    except Exception as e:
        out['motion_status']=type(e).__name__
    return out

if not MOTION.exists():
    rows=[]
    with ThreadPoolExecutor(max_workers=20) as ex:
        futures=[ex.submit(motion_one,r) for r in idx.itertuples(index=False)]
        for i,f in enumerate(as_completed(futures),1):
            rows.append(f.result())
            if i%50==0: print('motion',i,'/',len(futures),flush=True)
    pd.DataFrame(rows).to_csv(MOTION,index=False)
motion=pd.read_csv(MOTION,dtype={'subject_id':str})
primary=primary.merge(motion,on='subject_id',how='left')
primary.to_csv(OUT/'locked_primary_cohort_409_with_motion.csv',index=False)
display(primary.groupby(['site','label'])[['mean_fd','pct_fd_gt_0p2','mean_dvars','n_volumes']].mean())
print(primary.motion_status.value_counts(dropna=False))

## 2. Reusable fMRI feature bank

Features are cached so model comparisons use identical inputs: full A424 Fisher-z functional-connectivity edges, lower-dimensional ROI connectivity summaries, and ROI spectral/variability features.

In [ ]:
FEATURES=OUT/'a424_feature_bank.npz'
if not FEATURES.exists():
    tri=np.triu_indices(424,1)
    edges=[]; fc_summary=[]; spectral=[]
    for i,sid in enumerate(primary.subject_id,1):
        ts=np.load(TS_DIR/f'{sid}.npy').astype(np.float32)
        ts=np.nan_to_num(ts)
        c=np.corrcoef(ts,rowvar=False)
        c=np.nan_to_num(c); np.fill_diagonal(c,0)
        z=np.arctanh(np.clip(c,-0.999999,0.999999)).astype(np.float32)
        edges.append(z[tri])
        fc_summary.append(np.r_[z.mean(1),z.std(1)].astype(np.float32))
        f,p=periodogram(ts,fs=1.0,axis=0)
        band=p[(f>=0.01)&(f<=0.10)].sum(0)
        total=p[(f>0)&(f<=0.25)].sum(0)+1e-8
        falff=(band/total).astype(np.float32)
        spectral.append(np.r_[falff,ts.std(0)].astype(np.float32))
        if i%50==0: print('features',i,'/',len(primary),flush=True)
    np.savez_compressed(FEATURES,subject_id=primary.subject_id.to_numpy(),
                        fc_edges=np.stack(edges),fc_summary=np.stack(fc_summary),
                        spectral=np.stack(spectral))
bank=np.load(FEATURES,allow_pickle=True)
assert np.array_equal(bank['subject_id'].astype(str),primary.subject_id.to_numpy())
X_edges=bank['fc_edges']; X_fcsummary=bank['fc_summary']; X_spectral=bank['spectral']
emb_meta=pd.read_csv(BRAINLM/'brainlm_subject_embedding_metadata.csv',dtype={'subject_id':str})
emb=np.load(BRAINLM/'brainlm_subject_embeddings.npy')
emap={s:i for i,s in enumerate(emb_meta.subject_id.astype(str))}
X_brainlm=np.stack([emb[emap[s]] for s in primary.subject_id])
print('Edges',X_edges.shape,'FC summary',X_fcsummary.shape,
      'Spectral',X_spectral.shape,'BrainLM',X_brainlm.shape)

## 3. Nested LOSO benchmark

For every outer held-out site, logistic-regression C is selected using inner leave-one-training-site-out AUC. High-dimensional FC edges use a training-only univariate selector. No test-site labels or feature statistics enter fitting.

In [ ]:
def transform_fold(Xtr,ytr,Xte,k=None):
    imp=SimpleImputer(strategy='median').fit(Xtr)
    a=imp.transform(Xtr); b=imp.transform(Xte)
    if k is not None and a.shape[1]>k:
        sel=SelectKBest(f_classif,k=k).fit(a,ytr)
        a=sel.transform(a); b=sel.transform(b)
    sc=StandardScaler().fit(a)
    return sc.transform(a),sc.transform(b)

def nested_loso(X,name,k=None,Cs=(0.01,0.1,1.0)):
    y=primary.label.to_numpy(); g=primary.site.astype(str).to_numpy()
    logo=LeaveOneGroupOut(); oof=np.full(len(y),np.nan); rows=[]
    for tr,te in logo.split(X,y,g):
        scores={C:[] for C in Cs}
        for itr,iva in logo.split(X[tr],y[tr],g[tr]):
            a,b=transform_fold(X[tr][itr],y[tr][itr],X[tr][iva],k)
            for C in Cs:
                clf=LogisticRegression(C=C,class_weight='balanced',max_iter=3000)
                clf.fit(a,y[tr][itr])
                scores[C].append(roc_auc_score(y[tr][iva],clf.predict_proba(b)[:,1]))
        best=max(Cs,key=lambda C:np.nanmean(scores[C]))
        a,b=transform_fold(X[tr],y[tr],X[te],k)
        clf=LogisticRegression(C=best,class_weight='balanced',max_iter=3000)
        clf.fit(a,y[tr]); p=clf.predict_proba(b)[:,1]; oof[te]=p
        rows.append({'model':name,'site':g[te][0],'n':len(te),
                     'auc':roc_auc_score(y[te],p),'C':best})
    by=pd.DataFrame(rows)
    summary={'model':name,'n':len(y),'macro_auc':by.auc.mean(),
             'weighted_macro_auc':np.average(by.auc,weights=by.n),
             'pooled_oof_auc':roc_auc_score(y,oof)}
    pred=pd.DataFrame({'subject_id':primary.subject_id,'site':g,'y':y,'prob':oof,'model':name})
    return summary,by,pred

feature_sets={
    'age_sex':(primary[['age','sex_male']].to_numpy(float),None),
    'motion_qc':(primary[['mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy(float),None),
    'age_sex_motion':(primary[['age','sex_male','mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy(float),None),
    'fc_roi_summary':(X_fcsummary,None),
    'spectral_falff_variability':(X_spectral,None),
    'brainlm_frozen':(X_brainlm,None),
    'fc_edges_top2000':(X_edges,2000),
    'fc_summary_brainlm':(np.c_[X_fcsummary,X_brainlm],None),
}
summaries=[]; site_rows=[]; predictions=[]
for name,(X,k) in feature_sets.items():
    print('Running',name,flush=True)
    s,b,p=nested_loso(np.asarray(X,dtype=np.float32),name,k=k)
    summaries.append(s); site_rows.append(b); predictions.append(p)
    print(s,flush=True)
summary=pd.DataFrame(summaries).sort_values('macro_auc',ascending=False)
by_site=pd.concat(site_rows,ignore_index=True)
pred=pd.concat(predictions,ignore_index=True)
summary.to_csv(OUT/'benchmark_summary.csv',index=False)
by_site.to_csv(OUT/'benchmark_by_site.csv',index=False)
pred.to_csv(OUT/'benchmark_predictions.csv',index=False)
display(summary); display(by_site.pivot(index='site',columns='model',values='auc'))

## 4. Incremental value beyond confounds

These comparisons add each imaging representation to the age, sex, motion, scan-length, and coarse-QC baseline. An imaging representation has useful incremental value only if the combined model consistently exceeds the confound-only model across held-out sites.

In [ ]:
X_conf=primary[['age','sex_male','mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy(float)
incremental_sets={
    'confounds_plus_fc_summary':np.c_[X_conf,X_fcsummary],
    'confounds_plus_spectral':np.c_[X_conf,X_spectral],
    'confounds_plus_brainlm':np.c_[X_conf,X_brainlm],
    'confounds_plus_fc_brainlm':np.c_[X_conf,X_fcsummary,X_brainlm],
}
inc_s=[]; inc_b=[]; inc_p=[]
for name,X in incremental_sets.items():
    print('Running',name,flush=True)
    s,b,p=nested_loso(np.asarray(X,dtype=np.float32),name)
    inc_s.append(s); inc_b.append(b); inc_p.append(p)
    print(s,flush=True)
summary2=pd.concat([summary,pd.DataFrame(inc_s)],ignore_index=True).sort_values('macro_auc',ascending=False)
by_site2=pd.concat([by_site,*inc_b],ignore_index=True)
pred2=pd.concat([pred,*inc_p],ignore_index=True)
summary2.to_csv(OUT/'benchmark_summary_with_incremental.csv',index=False)
by_site2.to_csv(OUT/'benchmark_by_site_with_incremental.csv',index=False)
pred2.to_csv(OUT/'benchmark_predictions_with_incremental.csv',index=False)
display(summary2)

## 5. Motion-restricted sensitivity analysis

This secondary analysis repeats the strict nested LOSO benchmark after excluding subjects with missing motion estimates, mean framewise displacement above 0.30 mm, or more than 50% of frames above 0.20 mm. The thresholds are fixed before model evaluation. This is a sensitivity analysis rather than the primary result because motion-based exclusion can change the clinical composition of the cohort.

In [ ]:
sens_mask=(primary.motion_status.eq('ok') &
           (primary.mean_fd<=0.30) &
           (primary.pct_fd_gt_0p2<=0.50)).to_numpy()
print('QC-restricted n =',int(sens_mask.sum()),'/',len(primary))
display(primary.loc[sens_mask].groupby(['site','label']).size().unstack(fill_value=0))

primary_all=primary
primary=primary_all.loc[sens_mask].reset_index(drop=True)
Xfc=X_fcsummary[sens_mask]; Xsp=X_spectral[sens_mask]; Xbl=X_brainlm[sens_mask]
Xconf=primary[['age','sex_male','mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy(float)
sens_sets={
    'age_sex_motion_qc_restricted':Xconf,
    'brainlm_frozen_qc_restricted':Xbl,
    'fc_roi_summary_qc_restricted':Xfc,
    'spectral_falff_variability_qc_restricted':Xsp,
    'confounds_plus_brainlm_qc_restricted':np.c_[Xconf,Xbl],
    'confounds_plus_fc_summary_qc_restricted':np.c_[Xconf,Xfc],
    'confounds_plus_spectral_qc_restricted':np.c_[Xconf,Xsp],
}
sens_s=[]; sens_b=[]; sens_p=[]
try:
    for name,X in sens_sets.items():
        print('Running',name,flush=True)
        s,b,p=nested_loso(np.asarray(X,dtype=np.float32),name)
        sens_s.append(s); sens_b.append(b); sens_p.append(p)
        print(s,flush=True)
    sens_summary=pd.DataFrame(sens_s).sort_values('macro_auc',ascending=False)
    sens_by_site=pd.concat(sens_b,ignore_index=True)
    sens_pred=pd.concat(sens_p,ignore_index=True)
    sens_summary.to_csv(OUT/'benchmark_qc_restricted_summary.csv',index=False)
    sens_by_site.to_csv(OUT/'benchmark_qc_restricted_by_site.csv',index=False)
    sens_pred.to_csv(OUT/'benchmark_qc_restricted_predictions.csv',index=False)
    primary.to_csv(OUT/'locked_qc_restricted_cohort_351.csv',index=False)
    display(sens_summary)
    display(sens_by_site.pivot(index='site',columns='model',values='auc'))
finally:
    primary=primary_all

## 6. Continuous symptom-dimension benchmark

The ADHD-200 phenotype file uses `-999` for missing scores, and Peking uses a different score range. To avoid mistaking instrument/site differences for brain signal, this exploratory strict-LOSO regression uses only the comparable Inattentive and Hyper/Impulsive T-scores from KKI, NYU, and OHSU. ADHD Index is not evaluated because only two comparable sites are available.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

def transform_reg(Xtr,ytr,Xte):
    imp=SimpleImputer(strategy='median').fit(Xtr)
    a=imp.transform(Xtr); b=imp.transform(Xte)
    sc=StandardScaler().fit(a)
    return sc.transform(a),sc.transform(b)

def nested_loso_reg(X,y,g,sids,name,alphas=(1.,10.,100.,1000.)):
    logo=LeaveOneGroupOut(); oof=np.full(len(y),np.nan); rows=[]
    for tr,te in logo.split(X,y,g):
        scores={a:[] for a in alphas}
        for itr,iva in logo.split(X[tr],y[tr],g[tr]):
            xa,xb=transform_reg(X[tr][itr],y[tr][itr],X[tr][iva])
            for a in alphas:
                p=Ridge(alpha=a).fit(xa,y[tr][itr]).predict(xb)
                scores[a].append(np.sqrt(np.mean((y[tr][iva]-p)**2)))
        best=min(alphas,key=lambda a:np.mean(scores[a]))
        xa,xb=transform_reg(X[tr],y[tr],X[te])
        p=Ridge(alpha=best).fit(xa,y[tr]).predict(xb); oof[te]=p
        rows.append({'model':name,'site':g[te][0],'n':len(te),
                     'mae':mean_absolute_error(y[te],p),
                     'rmse':np.sqrt(np.mean((y[te]-p)**2)),
                     'r2':r2_score(y[te],p),
                     'pearson_r':pearsonr(y[te],p).statistic,
                     'spearman_r':spearmanr(y[te],p).statistic,'alpha':best})
    by=pd.DataFrame(rows)
    summary={'model':name,'n':len(y),'macro_pearson_r':by.pearson_r.mean(),
             'weighted_pearson_r':np.average(by.pearson_r,weights=by.n),
             'pooled_pearson_r':pearsonr(y,oof).statistic,
             'pooled_spearman_r':spearmanr(y,oof).statistic,
             'pooled_mae':mean_absolute_error(y,oof)}
    pred=pd.DataFrame({'subject_id':sids,'site':g,'y':y,'prediction':oof,'model':name})
    return summary,by,pred

sym_s=[]; sym_b=[]; sym_p=[]
for target in ['Inattentive','Hyper/Impulsive']:
    y0=pd.to_numeric(primary[target],errors='coerce')
    mask=(primary.site.isin(['KKI','NYU','OHSU']) & y0.notna() & (y0>-100)).to_numpy()
    sub=primary.loc[mask].reset_index(drop=True)
    y=y0.loc[mask].to_numpy(float); g=sub.site.to_numpy(str); sids=sub.subject_id.to_numpy(str)
    conf=sub[['age','sex_male','mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy(float)
    sets={'confounds':conf,'brainlm':X_brainlm[mask],'fc_summary':X_fcsummary[mask],
          'spectral':X_spectral[mask],
          'confounds_plus_brainlm':np.c_[conf,X_brainlm[mask]],
          'confounds_plus_fc_summary':np.c_[conf,X_fcsummary[mask]],
          'confounds_plus_spectral':np.c_[conf,X_spectral[mask]]}
    print('TARGET',target,'n',len(sub),sub.groupby('site').size().to_dict(),flush=True)
    for short,X in sets.items():
        name=target+'__'+short
        s,b,p=nested_loso_reg(np.asarray(X,dtype=np.float32),y,g,sids,name)
        s['target']=target; b['target']=target; p['target']=target
        sym_s.append(s); sym_b.append(b); sym_p.append(p)
sym_summary=pd.DataFrame(sym_s).sort_values(['target','macro_pearson_r'],ascending=[True,False])
sym_by_site=pd.concat(sym_b,ignore_index=True); sym_pred=pd.concat(sym_p,ignore_index=True)
sym_summary.to_csv(OUT/'symptom_loso_summary.csv',index=False)
sym_by_site.to_csv(OUT/'symptom_loso_by_site.csv',index=False)
sym_pred.to_csv(OUT/'symptom_loso_predictions.csv',index=False)
display(sym_summary)

## Interpretation gate

Proceed to adapter/last-block BrainLM fine-tuning only if an imaging representation is consistently above confound-only baselines and improves across several held-out sites. Otherwise, prioritize motion/site robustness, interpretable network features, and symptom-dimension analyses rather than increasing model capacity.